# T30B / E13 — Các lớp không tràn số của Sailor2-8B có cho giá trị đúng không

**Câu hỏi.** Với Qwen2.5-7B, T07 đã đối chiếu `float16` với `float32`: 27 lớp còn lại lệch trung bình
0,07 % thang đo. Với Sailor2-8B chưa có con số tương đương, vì lượt đối chiếu hết bộ nhớ hai lần khi hai
mô hình cùng nằm trong một tiến trình.

**Cách làm.** Mỗi kiểu số chạy trong **một tiến trình riêng**, ghi kết quả ra tệp, rồi so hai tệp. Đối chiếu
với `bfloat16` (cùng dải giá trị với `float32`, tốn bộ nhớ như `float16`). Vì `bfloat16` có ít chữ số có nghĩa
hơn, Qwen2.5-7B cũng được chạy theo đúng cách này để biết mức lệch nào là bình thường.

**Cách đọc.** Nếu độ lệch của Sailor2 cùng cỡ với Qwen2.5-7B thì các lớp còn lại dùng được; nếu lớn hơn
nhiều thì kết quả E13 phải kèm cảnh báo.

**Dự đoán ghi trước khi chạy:** độ lệch trung bình của Sailor2 không quá hai lần của Qwen2.5-7B.

**Chạy trong một phiên T4 riêng.** Ước tính 1–1,5 giờ, phần lớn là tải mô hình.

In [1]:
# Ô 1 — lấy mã nguồn. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

/kaggle/working/vihallulens
commit: e7f0335 Merge pull request #21 from wsunicorn/t45e-chi-phi-tu-kiem-tra


In [2]:
# Ô 2 — cài đặt, ghim phiên bản theo uv.lock.
!pip install -q --no-deps -e .
!pip install -q -r requirements-kaggle.txt

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.


In [3]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
import importlib.util
import sys
from pathlib import Path

import torch

sys.path.insert(0, "src")
from vihallulens.data.paths import find_raw_dir

problems = []
absent = [n for n in ("torch", "transformers", "bitsandbytes", "accelerate", "pandas")
          if importlib.util.find_spec(n) is None]
if absent:
    problems.append(f"thiếu gói {absent}")
if not torch.cuda.is_available():
    problems.append("chưa bật GPU T4")
else:
    print("GPU:", torch.cuda.get_device_name(0))
try:
    raw = Path(find_raw_dir())
    missing = [n for n in ['vihallu_train.csv', 'isedsc01_train.json'] if not (raw / n).is_file()]
    print("dữ liệu thô:", raw, "— thiếu", missing or "không")
    if missing:
        problems.append(f"thiếu tệp {missing}")
except Exception as error:
    problems.append(f"chưa gắn dataset dữ liệu thô ({error})")
if problems:
    raise SystemExit("TIỀN KIỂM HỎNG: " + "; ".join(problems))
print("Tiền kiểm đạt.")

GPU: Tesla T4
dữ liệu thô: /kaggle/input/datasets/unicorn1209/vihallulens — thiếu không
Tiền kiểm đạt.


In [4]:
# Ô 4 — kiểm môi trường. Một phút, CPU.
!python scripts/probe_env.py
!python -m pytest tests/test_compare_dtypes.py tests/test_attention_hook.py tests/test_revisions.py -q
!df -h /root | tail -1


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : e7f0335 Merge pull request #21 from wsunicorn/t45e-chi-phi-tu-kiem-tra
  python           : 3.13.16
  torch            : 2.11.0+cu130
  transformers     : 5.17.0
  bitsandbytes     : 0.50.2
  accelerate       : 1.15.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv
........................                                                 [100%]
overlay         8.0T  5.6T  2.5T  70% /


In [5]:
# Ô 5 — bốn lượt, mỗi lượt một tiến trình. Một lượt hỏng KHÔNG dừng các lượt sau.
# Trọng số Sailor2 được xóa khỏi đĩa của phiên Kaggle sau khi dùng, để đủ chỗ tải Qwen2.5-7B.
import os
import shutil
import sys
from pathlib import Path

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
OUT = Path("/kaggle/working/t30b")
OUT.mkdir(exist_ok=True)
MODELS = {"sailor2": "sail/Sailor2-8B-SFT", "qwen7b": "Qwen/Qwen2.5-7B-Instruct"}
status = {}
for key, model in MODELS.items():
    for dtype in ("float16", "bfloat16"):
        target = OUT / f"{key}_{dtype}.npz"
        if target.is_file():
            status[key, dtype] = "đã có"
            continue
        print("=" * 78)
        print(f"  {model} ở {dtype}", flush=True)
        done = subprocess.run([sys.executable, "scripts/compare_dtypes.py", "--model", model,
                               "--dtype", dtype, "--save", str(target)],
                              capture_output=True, text=True)
        print(done.stdout[-3500:])
        if done.returncode:
            print(done.stderr[-2500:])
        status[key, dtype] = "xong" if target.is_file() else f"HỎNG (mã {done.returncode})"
    hub = Path.home() / ".cache/huggingface/hub" / ("models--" + model.replace("/", "--"))
    if key == "sailor2" and all((OUT / f"{key}_{d}.npz").is_file()
                                for d in ("float16", "bfloat16")):
        shutil.rmtree(hub, ignore_errors=True)
for (key, dtype), value in status.items():
    print(f"  {key:<8} {dtype:<9} {value}")

  sail/Sailor2-8B-SFT ở float16

SO SÁNH FLOAT16 VỚI NONE
  dữ liệu   : /kaggle/input/datasets/unicorn1209/vihallulens
  số mẫu    : 20
  độ dài    : 47 đến 4805 từ
    [float16] 1/20
    [float16] 2/20
    [float16] 3/20
    [float16] 4/20
    [float16] 5/20
    [float16] 6/20
    [float16] 7/20
    [float16] 8/20
    [float16] 9/20
    [float16] 10/20
    [float16] 11/20
    [float16] 12/20
    [float16] 13/20
    [float16] 14/20
    [float16] 15/20
    [float16] 16/20
    [float16] 17/20
    [float16] 18/20
    [float16] 19/20
    [float16] 20/20
    [float16] xong 20 mẫu trong 21 s
  đã ghi /kaggle/working/t30b/sailor2_float16.npz

  float16  :     1015 ms mỗi mẫu, VRAM đỉnh     8785 MB

  Mẫu có ít nhất một lớp nan ở float16  : 20/20
  Tập hợp các lớp từng nan          : [30, 31]
EXCLUDE_LAYERS=[30, 31]
N_LAYERS=32

  Không có lượt mốc để so, nên bỏ bảng theo lớp và phần |Δ|.
  Phần trên là đủ để điền exclude_layers; phần thiếu chỉ trả lời câu hỏi thứ hai
  của script — các lớp cò

In [6]:
# Ô 6 — so sánh. Cần đủ hai tệp của một mô hình; thiếu thì báo và bỏ qua mô hình đó.
import subprocess
import sys
from pathlib import Path

OUT = Path("/kaggle/working/t30b")
for key in ("qwen7b", "sailor2"):
    low, high = OUT / f"{key}_float16.npz", OUT / f"{key}_bfloat16.npz"
    print("=" * 78)
    if not (low.is_file() and high.is_file()):
        print(f"  {key}: thiếu tệp, không so được")
        continue
    done = subprocess.run([sys.executable, "scripts/compare_dtypes.py", "--from-saved",
                           str(low), str(high)], capture_output=True, text=True)
    print(done.stdout or done.stderr)
    (OUT / f"{key}_so_sanh.txt").write_text(done.stdout, encoding="utf-8")
!ls -la /kaggle/working/t30b

Qwen/Qwen2.5-7B-Instruct: float16 so với bfloat16, 20 mẫu, 28 lớp
  float16  : lớp từng tràn số [27]
  bfloat16 : lớp từng tràn số []
EXCLUDE_LAYERS=[27]

   Lớp |         mẫu nan |  |Δ| trung bình |  |Δ| lớn nhất
  -----+-----------------+-----------------+--------------
     0 |       0/20      |         0.02566 |       1.00000
     1 |       0/20      |         0.01263 |       0.60522
     2 |       0/20      |         0.01555 |       0.55103
     3 |       0/20      |         0.01210 |       0.63782
     4 |       0/20      |         0.01910 |       0.69821
     5 |       0/20      |         0.02593 |       0.67615
     6 |       0/20      |         0.02065 |       0.73108
     7 |       0/20      |         0.02052 |       0.67871
     8 |       0/20      |         0.02085 |       0.88800
     9 |       0/20      |         0.02585 |       0.84631
    10 |       0/20      |         0.02121 |       0.82751
    11 |       0/20      |         0.02542 |       0.89935
    12 |       0/20

In [7]:
# Ô 7 — gói lại để tải về một tệp: /kaggle/working/t30b.zip
import shutil

print(shutil.make_archive("/kaggle/working/t30b", "zip", "/kaggle/working/t30b"))

/kaggle/working/t30b.zip
